# Luminancia

## Media de luminancia

Promedia los valores de luminancia de todos los píxeles para obtener el nivel de brillo general de la imagen.

In [1]:
from skimage import io
import numpy as np

def obtener_media_luminancia(img_path):
    return np.mean(io.imread(img_path, as_gray=True))

## Varianza de luminancia

Mide qué tan uniformemente distribuida está la luminancia. Valores altos = alto contraste; valores bajos = uniformidad.

In [2]:
from skimage import io
import numpy as np

def obtener_varianza_luminancia(img_path):
    return np.var(io.imread(img_path, as_gray=True))

## Histograma de intensidades

Frecuencia de cada nivel de intensidad (0–255) en la imagen. Base para diagnóstico de exposición y contraste.

In [3]:
from skimage import io
import numpy as np

def obtener_histograma_intensidades(img_path):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    hist = np.bincount(img.ravel(), minlength=256)
    return hist, hist / hist.sum()

## Rango dinámico

Diferencia entre el valor máximo y mínimo de luminancia. Indica qué tan bien se captura el espectro completo de brillos.

In [4]:
from skimage import io
import numpy as np

def obtener_rango_dinamico(img_path):
    return np.ptp(io.imread(img_path, as_gray=True))

## Desviación estándar de brillo

Raíz cuadrada de la varianza de luminancia. Representa el contraste global típico de la imagen.

In [5]:
from skimage import io
import numpy as np

def obtener_desv_estandar_brillo(img_path):
    return np.std(io.imread(img_path, as_gray=True))

## Contraste local basado en luminancia

Compara la luminancia de cada píxel con la media de su vecindario local, normalizada por dicha media.

In [6]:
from skimage import io
from scipy.ndimage import uniform_filter

def obtener_contraste_local_luminancia(img_path, ventana=5):
    img = io.imread(img_path, as_gray=True)
    mu = uniform_filter(img, ventana)
    return (img - mu) / (mu + 1e-8)

# Contraste 

## Contraste de Michelson

Contraste modular entre máximo y mínimo de luminancia en una región. Adimensional (0 a 1), robusto ante cambios globales de brillo.

In [7]:
from skimage import io
import numpy as np

def obtener_contraste_michelson(img_path):
    img = io.imread(img_path, as_gray=True)
    
    Lmax = np.percentile(img, 98)
    Lmin = np.percentile(img, 2)
    
    return (Lmax - Lmin) / (Lmax + Lmin + 1e-8)

## Contraste local (revisar)

Mide la diferencia de luminancia entre un píxel y su vecindario inmediato. Útil para detectar bordes y texturas.

In [8]:
from skimage import io
from scipy.ndimage import uniform_filter
import numpy as np

def contraste_local(img_path, ventana=5):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    mu = uniform_filter(img, ventana)
    return (img - mu) / (mu + 1e-8)

## Desviación estándar de intensidad

Dispersión de valores de intensidad en la imagen. Indicador de contraste y variabilidad textural.

In [9]:
from skimage import io
import numpy as np

def obtener_desv_estandar_intensidad(img_path):
    return np.std(io.imread(img_path, as_gray=True).astype(np.float32))

# Entropía (Contenido de Información)


## Entropía de Shannon

Mide la incertidumbre o cantidad de información promedio en la distribución de intensidades. Indicador de complejidad.

In [10]:
from skimage import io
from skimage.measure import shannon_entropy
import numpy as np

def obtener_entropia_shannon(img_path):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    return shannon_entropy(img)

## Entropía conjunta

Información total de incertidumbre asociada a dos variables (canales o imágenes) consideradas simultáneamente.

In [11]:
from skimage import io
import numpy as np

def entropia_conjunta(img1_path, img2_path):
    img1 = (io.imread(img1_path, as_gray=True) * 255).astype(np.uint8).ravel()
    img2 = (io.imread(img2_path, as_gray=True) * 255).astype(np.uint8).ravel()
    
    hist2d, _, _ = np.histogram2d(img1, img2, bins=256, range=[[0,256],[0,256]])
    prob = hist2d / hist2d.sum()
    
    prob = prob[prob > 0]
    return -np.sum(prob * np.log2(prob))

## Entropía local por bloques

Entropía de Shannon calculada en bloques locales de la imagen. Permite analizar variabilidad espacial de la información.

In [12]:
from skimage import io
import numpy as np

def entropia_local_bloques(img_path, bloque=16):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    M, N = img.shape
    
    img = img[:M - M % bloque, :N - N % bloque]  # recorte exacto
    bloques = img.reshape(M//bloque, bloque, N//bloque, bloque).swapaxes(1,2)
    
    mapa = np.zeros((M//bloque, N//bloque))
    
    for i in range(mapa.shape[0]):
        for j in range(mapa.shape[1]):
            b = bloques[i, j].ravel()
            hist = np.bincount(b, minlength=256)
            p = hist / hist.sum()
            p = p[p > 0]
            mapa[i, j] = -np.sum(p * np.log2(p))
    
    return mapa

# Frecuencias

## Energía en altas frecuencias

Concentración de potencia espectral en componentes de alta frecuencia. Indica presencia de bordes y detalles nítidos.

In [13]:
from skimage import io
import numpy as np

def obtener_energia_altas_frecuencias(img_path, radio=0.5):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    F = np.fft.fftshift(np.fft.fft2(img))
    
    M, N = img.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]
    
    mask = (X - cx)**2 + (Y - cy)**2 > (radio * min(M, N) / 2)**2
    return np.sum(np.abs(F[mask])**2)

## Energía en bajas frecuencias

Potencia espectral en componentes de baja frecuencia. Corresponde a variaciones suaves, fondos y formas generales.

In [14]:
from skimage import io
import numpy as np

def obtener_energia_bajas_frecuencias(img_path, radio=0.5):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    F = np.fft.fftshift(np.fft.fft2(img))
    
    M, N = img.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]
    
    mask = (X - cx)**2 + (Y - cy)**2 <= (radio * min(M, N) / 2)**2
    return np.sum(np.abs(F[mask])**2)

## Relación high/low frequency ratio

Proporción de energía entre altas y bajas frecuencias. Evalúa el nivel de detalle o nitidez frente a la estructura global.

In [15]:
from skimage import io
import numpy as np

def obtener_hlfr(img_path, radio=0.5):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    F = np.fft.fftshift(np.fft.fft2(img))
    
    M, N = img.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]
    
    mask = (X - cx)**2 + (Y - cy)**2 > (radio * min(M, N) / 2)**2
    
    EHF = np.sum(np.abs(F[mask])**2)
    ELF = np.sum(np.abs(F[~mask])**2)
    
    return EHF / (ELF + 1e-8)

## Densidad espectral de potencia 

Distribución de la energía de la imagen en el dominio de frecuencias 2D. Se obtiene de la DFT bidimensional.

In [16]:
from skimage import io
import numpy as np

def psd(img_path):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    M, N = img.shape
    F = np.fft.fft2(img)
    return (np.abs(F)**2) / (M * N)

# Bordes y Gradientes

## Densidad de bordes

Fracción de píxeles detectados como bordes respecto al total de píxeles de la imagen.

In [17]:
from skimage import io, feature
import numpy as np

def obtener_densidad_bordes(img_path, sigma=1.0):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    edges = feature.canny(img, sigma=sigma)
    return edges.mean()

## Magnitud media del gradiente

Cambio promedio de intensidad en la imagen. Indica el nivel global de bordes y detalles presentes.

In [18]:
from skimage import io
from scipy.ndimage import sobel
import numpy as np

def obtener_magnitud_media_gradiente(img_path):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    Gx = sobel(img, axis=1)
    Gy = sobel(img, axis=0)
    return np.mean(np.hypot(Gx, Gy))

## Desviación estándar del gradiente

Variabilidad de la magnitud del gradiente. Cuantifica la complejidad estructural de la imagen.

In [19]:
from skimage import io
from scipy.ndimage import sobel
import numpy as np

def obtener_desv_estandar_gradiente(img_path):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    Gx = sobel(img, axis=1)
    Gy = sobel(img, axis=0)
    return np.std(np.hypot(Gx, Gy))

## Mapa Sobel

Calcula gradientes horizontales y verticales mediante convolución con máscaras Sobel. Produce bordes gruesos.

In [20]:
from skimage import io
from scipy.ndimage import sobel
import numpy as np

def mapa_sobel(img_path):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    Gx = sobel(img, axis=1)
    Gy = sobel(img, axis=0)
    return np.hypot(Gx, Gy)

## Mapa Canny

Detector óptimo multi-etapa: suavizado gaussiano → gradiente → supresión de no-máximos → histéresis.

In [21]:
from skimage import io, feature
import numpy as np

def mapa_canny(img_path, sigma=1.0, low=0.1, high=0.2):
    img = io.imread(img_path, as_gray=True).astype(np.float32)
    edges = feature.canny(img, sigma=sigma,
                          low_threshold=low, high_threshold=high)
    return edges.astype(np.float32)

# Textura

## Haralick features (GLCM)

14 medidas estadísticas de textura de segundo orden basadas en la matriz de co-ocurrencia de niveles de gris.

In [22]:
from skimage import io
from skimage.feature import graycomatrix, graycoprops
import numpy as np

def haralick_features(img_path, distancias=[1], angulos=[0]):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    
    glcm = graycomatrix(img,
                         distances=distancias,
                         angles=angulos,
                         levels=256,
                         symmetric=True,
                         normed=True)
    
    props = ['contrast', 'dissimilarity', 'homogeneity',
             'energy', 'correlation', 'ASM']
    
    return {p: graycoprops(glcm, p)[0, 0] for p in props}

## Homogeneidad de textura (GLCM)

Qué tan cercanos son los valores de intensidad de píxeles vecinos. Alto = textura suave o uniforme.

In [23]:
from skimage import io
from skimage.feature import graycomatrix, graycoprops
import numpy as np

def obtener_homogeneidad_textura(img_path):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    
    glcm = graycomatrix(img,
                         distances=[1],
                         angles=[0],
                         levels=256,
                         symmetric=True,
                         normed=True)
    
    return graycoprops(glcm, 'homogeneity')[0, 0]

## Energía de textura (GLCM Energy / ASM)

Mide la uniformidad global (Angular Second Moment). Alto = textura uniforme o repetitiva.

In [24]:
from skimage import io
from skimage.feature import graycomatrix, graycoprops
import numpy as np

def obtener_energia_textura(img_path):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    
    glcm = graycomatrix(img,
                         distances=[1],
                         angles=[0],
                         levels=256,
                         symmetric=True,
                         normed=True)
    
    return graycoprops(glcm, 'energy')[0, 0]

## Correlación (GLCM)

Dependencia lineal entre niveles de gris de píxeles vecinos. Alta correlación = patrón estructural consistente.

In [25]:
from skimage import io
from skimage.feature import graycomatrix, graycoprops
import numpy as np

def obtener_correlacion_glcm(img_path):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    
    glcm = graycomatrix(img,
                         distances=[1],
                         angles=[0],
                         levels=256,
                         symmetric=True,
                         normed=True)
    
    return graycoprops(glcm, 'correlation')[0, 0]

## Contraste de textura (GLCM Contrast)

Variaciones locales de intensidad entre píxeles vecinos. Cuantifica la aspereza o definición de bordes en la textura.

In [26]:
from skimage import io
from skimage.feature import graycomatrix, graycoprops
import numpy as np

def obtener_contraste_textura(img_path):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    
    glcm = graycomatrix(img,
                         distances=[1],
                         angles=[0],
                         levels=256,
                         symmetric=True,
                         normed=True)
    
    return graycoprops(glcm, 'contrast')[0, 0]

## Entropía de textura (GLCM Entropy)

Grado de aleatoriedad o desorden en la distribución espacial de niveles de gris entre píxeles vecinos.

In [27]:
from skimage import io
from skimage.feature import graycomatrix
import numpy as np

def obtener_entropia_textura_glcm(img_path):
    img = (io.imread(img_path, as_gray=True) * 255).astype(np.uint8)
    
    glcm = graycomatrix(img,
                         distances=[1],
                         angles=[0],
                         levels=256,
                         symmetric=True,
                         normed=True)
    
    P = glcm[:, :, 0, 0]
    P_nz = P[P > 0]
    
    return -np.sum(P_nz * np.log2(P_nz))

# Saturación y Crominancia

## Media de saturación

Nivel promedio de pureza cromática en la imagen en espacio HSV.

In [28]:
from skimage import color

def safe_rgb(img):
    if img.ndim == 2:  # grayscale
        return color.gray2rgb(img)
    if img.shape[-1] == 4:
        return color.rgba2rgb(img)
    return img

In [29]:
from skimage import io, color
import numpy as np

def obtener_media_saturacion(img_path):
    
    img = io.imread(img_path)
    img=safe_rgb(img)
    
    hsv = color.rgb2hsv(img)
    return hsv[:, :, 1].mean()

## Varianza de saturación

Dispersión de valores de saturación. Indica la uniformidad o variabilidad cromática de la imagen.

In [30]:
from skimage import io, color
import numpy as np

def obtener_varianza_saturacion(img_path):
    img = io.imread(img_path)
    
    # asegurar RGB
    if img.shape[-1] == 4:
        img = color.rgba2rgb(img)
    
    hsv = color.rgb2hsv(img)
    S = hsv[:, :, 1]
    
    return np.var(S)

## Varianza de matiz (Hue variance)

Dispersión angular de los tonos de color dominantes. Indica la diversidad cromática en el espacio HSV.

In [31]:
from skimage import io, color
import numpy as np

def obtener_varianza_matiz(img_path):
    img = io.imread(img_path)
    
    # asegurar RGB
    if img.shape[-1] == 4:
        img = color.rgba2rgb(img)
    
    hsv = color.rgb2hsv(img)
    H = hsv[:, :, 0]
    
    return np.var(H)

## Distribución en espacio CIELAB

Dispersión de colores en el espacio perceptual Lab*, diseñado para coincidir con la visión humana.

In [32]:
from skimage import io, color
import numpy as np

def obtener_distribucion_cielab(img_path):
    img = io.imread(img_path)
    
    # asegurar RGB
    if img.shape[-1] == 4:
        img = color.rgba2rgb(img)
    
    lab = color.rgb2lab(img)
    L, a, b = lab[:, :, 0], lab[:, :, 1], lab[:, :, 2]
    
    return {
        'media_a': np.mean(a),
        'media_b': np.mean(b),
        'varianza_a': np.var(a),
        'varianza_b': np.var(b)
    }

# Resolución Espacial

## Megapíxeles

Resolución total de la imagen expresada en millones de píxeles.

In [33]:
from skimage import io

def obtener_megapixeles(img_path):
    img = io.imread(img_path)
    h, w = img.shape[:2]
    return (h * w) / 1e6

## Aspect ratio

Proporción entre el ancho y el alto de la imagen, expresada como razón W/H.

In [34]:
from skimage import io

def obtener_aspect_ratio(img_path):
    img = io.imread(img_path)
    h, w = img.shape[:2]
    return w / h